# RAG Eval — Шаг 1: Генерация тестового датасета Q&A

старая версия\
Для каждой заметки из `notes_dataset_v1.xlsx`:
1. Генерируем вопрос + эталонный ответ через `openai/gpt-5.1` (сильная модель).
2. Определяем эталонный набор zettel-ов (ground truth) для метрик retrieval:
   через cosine-similarity эмбеддингов zettel-ов с текстом заметки.
3. Сохраняем в `synthetic_datasets/rag_qa_dataset.xlsx`.

**User в Neo4j:** `linker_eval_gpt-5.2`  
**Источник zettel-ов:** `synthetic_datasets/linker/reference_linker_gemini-2.5-falsh.xlsx

новая версия пайплайна

``` 
ШАГ 1: Atomizer + Linker
────────────────────────
Для каждой заметки из df_notes:
  - Atomizer разбивает текст → [ZettelCard_1..N]
  - Linker вставляет в Neo4j → каждая карточка получает zettel_id
  - Сохраняем маппинг: note_id → [zettel_id_1, ..., zettel_id_N]
  - Сохраняем карточки: zettel_id → content (для следующего шага)

ШАГ 2: Генерация Q&A с разметкой (GPT видит заметку + карточки)
────────────────────────────────────────────────────────────────
Для каждой заметки:
  - GPT получает: note_text + список {zettel_id, content} всех карточек
  - GPT генерирует: вопрос + reference_answer + relevant_zettel_ids + question_type
  - relevant_zettel_ids = ground truth для retriever (выбраны самим GPT)
  - Сохраняем qa_dataset с колонками:
      qa_id, note_id, question, reference_answer, 
      question_type, key_concepts,
      all_zettel_ids, relevant_zettel_ids  ← ground truth

ШАГ 3: Запуск Retriever
────────────────────────
Для каждого вопроса из qa_dataset:
  - GraphRetriever.retrieve(user_id, question)
  - Получаем ranked список zettel_id: [entry_points] + [expanded]
  - Сохраняем: retrieved_zettel_ids_ranked

ШАГ 4: Расчёт метрик Retriever
────────────────────────────────
Для каждого вопроса:
  retrieved = [zettel_id_1, zettel_id_2, ...]  ← от Retriever (ранжированный)
  relevant  = {zettel_id_A, zettel_id_B}       ← ground truth от GPT
  
  Считаем: Precision@K, Recall@K, F1@K, Hit-Rate@K, MRR, NDCG@K
  Агрегируем: mean по всем вопросам + разбивка по question_type
  ```

```
note_id | note_text → [Atomizer] → [Linker/Neo4j] → zettel_id карточек
                                                              ↓
                              GPT видит: note_text + {zettel_id, content}[]
                                                              ↓
                              GPT выдаёт: question
                                          reference_answer     → для оценки Generator
                                          question_type
                                          key_concepts         → для оценки Generator
                                          relevant_zettel_ids  → GROUND TRUTH Retriever
                                                              ↓
                              qa_dataset_annotated.xlsx
                                                              ↓
                              [GraphRetriever.retrieve(question)]
                                                              ↓
                              retrieved_zettel_ids_ranked (entry_points + expanded)
                                                              ↓
                              compare(retrieved, relevant) → метрики
                              ```

In [ ]:
import os
import json
import uuid
import pandas as pd
from tqdm import tqdm
from typing import Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

from zettelkasten.atomizer import NoteAtomizer
from zettelkasten.linker import GraphLinker, LocalEmbeddingModel
from storage.neo4j.client import get_neo4j_client
from storage.neo4j.schema import init_schema
from storage.neo4j.repository import ZettelRepository
from zettelkasten.graph_rag import GraphRetriever

load_dotenv()

/Users/nastya/github/Executive_Exocortex/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Atomizer → Linker → Q&A генерация

In [ ]:
#   1. Прогоняем заметки через Atomizer + Linker → получаем zettel_id
#   2. GPT видит заметку + готовые карточки → генерирует Q&A с разметкой
  
# На выходе: qa_dataset_annotated.xlsx с полями:
#   - question, reference_answer, question_type, key_concepts
#   - all_zettel_ids       (все карточки заметки)
#   - relevant_zettel_ids  (ground truth для retriever) -->


In [ ]:
NOTES_PATH      = "synthetic_datasets/notes_dataset_v1.xlsx"
MAPPING_OUTPUT  = "synthetic_datasets/graphrag/note_to_zettel_mapping.json"
QA_OUTPUT       = "synthetic_datasets/graphrag/qa_dataset_annotated.xlsx"
TEST_USER_ID    = "graphrag_test_user_v9"
QA_MODEL        = "openai/gpt-5.1"  # модель для генерации Q&A


In [ ]:
class QAWithRelevance(BaseModel):
    question: str = Field(
        description=(
            "Натуральный вопрос топ-менеджера к своей базе знаний. "
            "Верхнеуровневый — как будто забыл детали, но помнит контекст. "
            "Специфичный: содержит имена, проекты, цифры из заметки. "
            "Не цитирует дословно текст заметки."
        )
    )
    reference_answer: str = Field(
        description=(
            "Полный эталонный ответ на вопрос, основанный ТОЛЬКО на тексте заметки. "
            "Это идеальный ответ RAG-системы, с которым будет сравниваться генератор."
        )
    )
    question_type: str = Field(
        description="Тип вопроса: factual / summary / action_items / risk / decision / detail"
    )
    key_concepts: list[str] = Field(
        description=(
            "3-7 ключевых концептов из заметки, которые ДОЛЖНЫ присутствовать "
            "в хорошем ответе. Используется для оценки Generator."
        )
    )
    relevant_zettel_ids: list[str] = Field(
        description=(
            "Список zettel_id карточек из предоставленного списка, "
            "которые содержат информацию для ответа на вопрос. "
            "Это ground truth для оценки Retriever. "
            "Выбирай только те карточки, без которых нельзя ответить на вопрос. "
            "Минимум 1, максимум — все нужные."
        )
    )

In [ ]:

QA_SYSTEM_PROMPT = """Ты — топ-менеджер, который пользуется персональной системой управления знаниями.
Тебе показывают одну из твоих старых заметок и список атомарных карточек,
на которые система разбила эту заметку.

Твоя задача:
1. Придумать ОДИН реалистичный вопрос, который ты мог бы задать системе СПУСТЯ НЕКОТОРОЕ ВРЕМЯ.
2. Написать эталонный ответ на этот вопрос.
3. Выбрать из списка карточек те zettel_id, которые содержат ответ на вопрос.

Требования к вопросу:
- Верхнеуровневый — как будто забыл детали, но помнит общий контекст
- Специфичный: содержит имена, проекты или цифры из заметки
- Имеет чёткий ответ в тексте заметки
- Не цитирует дословно заметку
- Не слишком широкий ("о чём эта заметка?") и не слишком узкий

Требования к relevant_zettel_ids:
- Включай ТОЛЬКО те карточки, которые напрямую отвечают на вопрос
- Не включай карточки с общим контекстом, если они не нужны для ответа
- Проверяй: если убрать эту карточку, станет ли ответ неполным? Если да — включай."""

QA_USER_PROMPT = """## Исходная заметка:
{note_text}

## Атомарные карточки (результат разбивки заметки системой):
{cards_block}

Сгенерируй вопрос, эталонный ответ и выбери relevant_zettel_ids."""


In [ ]:
def format_cards_block(cards: list[dict]) -> str:
    """Форматирует список карточек для промпта."""
    lines = []
    for i, card in enumerate(cards, 1):
        lines.append(
            f"{i}. zettel_id: {card['zettel_id']}\n"
            f"   Тип: {card['thought_type']}\n"
            f"   Содержание: {card['content']}"
        )
    return "\n\n".join(lines)

In [ ]:
class QAGenerator:
    def __init__(self, model_name: str = QA_MODEL):
        self.llm = ChatOpenAI(
            model=model_name,
            api_key=os.getenv("LLM_API_KEY"),
            base_url=os.getenv("LLM_BASE_URL"),
            temperature=0.7,
        ).with_structured_output(QAWithRelevance)

    def generate(
        self,
        note_text: str,
        cards: list[dict],  # [{"zettel_id": str, "content": str, "thought_type": str}]
    ) -> Optional[QAWithRelevance]:
        try:
            cards_block = format_cards_block(cards)
            result = self.llm.invoke([
                SystemMessage(content=QA_SYSTEM_PROMPT),
                HumanMessage(content=QA_USER_PROMPT.format(
                    note_text=note_text,
                    cards_block=cards_block,
                )),
            ])

            # валидация: все relevant_zettel_ids должны быть из списка карточек
            valid_ids = {c["zettel_id"] for c in cards}
            result.relevant_zettel_ids = [
                zid for zid in result.relevant_zettel_ids
                if zid in valid_ids
            ]

            return result
        except Exception as e:
            print(f"  [ERROR] QA generation failed: {e}")
            return None



In [ ]:
def build_graph_and_generate_qa():

    # загружаем заметки
    df_notes = pd.read_excel(NOTES_PATH)[:5]  # 100 заметок
    print(f"Заметок для обработки: {len(df_notes)}")

    # инициализация компонентов
    embedding_model = LocalEmbeddingModel()
    client = get_neo4j_client()
    init_schema(client)
    repository = ZettelRepository(client)
    atomizer = NoteAtomizer()
    linker = GraphLinker(embedding_model=embedding_model, repository=repository)
    qa_generator = QAGenerator()

    note_to_zettels: dict[str, list[str]] = {}  # note_id → [zettel_id]
    qa_rows = []

    for _, note_row in tqdm(df_notes.iterrows(), total=len(df_notes), desc="Обработка заметок"):
        note_id   = str(note_row["note_id"])
        note_text = str(note_row["note_text"])

        #─  Atomizer
        current_max_root = repository.get_max_root_id(TEST_USER_ID)
        cards = atomizer.atomize(note_text, current_db_max_root_id=current_max_root)

        if isinstance(cards, str):
            print(f"  [WARN] Atomizer error: {cards}")
            note_to_zettels[note_id] = []
            continue

        #  Linker → Neo4
        try:
            results = linker.link_and_insert(TEST_USER_ID, cards)
        except Exception as e:
            linker.model_name = "openai/gpt-5.1"
            results = linker.link_and_insert(TEST_USER_ID, cards)
        
        # собираем карточки с их zettel_id из Neo4j
        inserted_cards = []
        for res in results:
            if res.card and res.card.zettel_id:
                inserted_cards.append({
                    "zettel_id":    res.card.zettel_id,
                    "content":      res.card.content,
                    "thought_type": res.card.thought_type,
                    "luhmann_id":   res.card.luhmann_id,
                })

        zettel_ids = [c["zettel_id"] for c in inserted_cards]
        note_to_zettels[note_id] = zettel_ids
        # print(f"  ✅ Atomizer: {len(inserted_cards)} карточек вставлено")

        if not inserted_cards:
            continue

        # Генерация Q&A с разметкой ─────────────────────────────────
        qa = qa_generator.generate(note_text, inserted_cards)

        if qa is None:
            print(f"  [WARN] Q&A generation failed для note_id={note_id}")
            continue

        # print(f"  ✅ Q&A: '{qa.question[:60]}...'")
        # print(f"  ✅ Relevant cards: {qa.relevant_zettel_ids}")

        qa_rows.append({
            "qa_id":               str(uuid.uuid4()),
            "note_id":             note_id,
            "note_text":           note_text,
            "domain":              note_row.get("domain", ""),
            "style":               note_row.get("style", ""),
            # Q&A
            "question":            qa.question,
            "reference_answer":    qa.reference_answer,
            "question_type":       qa.question_type,
            "key_concepts":        "|".join(qa.key_concepts),
            # Ground truth для retriever
            "all_zettel_ids":      "|".join(zettel_ids),
            "relevant_zettel_ids": "|".join(qa.relevant_zettel_ids),  # ← GROUND TRUTH
            "n_total_zettels":     len(zettel_ids),
            "n_relevant_zettels":  len(qa.relevant_zettel_ids),
        })

    # сохраняем маппинг
    os.makedirs(os.path.dirname(MAPPING_OUTPUT), exist_ok=True)
    with open(MAPPING_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(note_to_zettels, f, ensure_ascii=False, indent=2)
    print(f"\nМаппинг → {MAPPING_OUTPUT}")

    # сохраняем Q&A датасет
    df_qa = pd.DataFrame(qa_rows)
    os.makedirs(os.path.dirname(QA_OUTPUT), exist_ok=True)
    df_qa.to_excel(QA_OUTPUT, index=False)

    print(f"Q&A датасет → {QA_OUTPUT}")
    print(f"Итого: {len(df_qa)} вопросов")
    print(df_qa[["question_type", "n_total_zettels", "n_relevant_zettels"]].describe())

    return df_qa


if __name__ == "__main__":
    build_graph_and_generate_qa()

Заметок для обработки: 10


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8453.35it/s]
/Users/nastya/github/Executive_Exocortex/zettelkasten/linker.py:84: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


[EmbeddingModel] intfloat/multilingual-e5-base загружена за 7.4с (dim=768, device=mps)
[Neo4j Schema] Инициализация схемы...
[Neo4j Schema] Vector index создан (dim=768, cosine)
[Neo4j Schema] Схема готова


Обработка заметок:   0%|          | 0/10 [00:00<?, ?it/s]Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `RELATED_TO` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=71, offset=71>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 71, 'line': 2, 'column': 71}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (z:Zettel {zettel_id: $zettel_id, user_id: $user_id})-[:RELATED_TO*1..1]-(related:Zettel {user_id: $user_id})\n        WHERE related.zettel_id <> $zettel_id\n        RETURN DISTINCT related as z\n        LIMIT 10\n        '
Received notification

  [Neo4j] Создан [1]: Рассматривается реструктуризация после разговора с...
  [Neo4j] Создан [1.1] ← [1]
  [Neo4j] Создан [1.1a] ← [1.1]


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `RELATED_TO` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=71, offset=71>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 71, 'line': 2, 'column': 71}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (z:Zettel {zettel_id: $zettel_id, user_id: $user_id})-[:RELATED_TO*1..1]-(related:Zettel {user_id: $user_id})\n        WHERE related.zettel_id <> $zettel_id\n        RETURN DISTINCT related as z\n        LIMIT 10\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N51', s

  [Neo4j] Создан [26]: Команда Phoenix выделит два дня на рефакторинг оче...
  [Neo4j] Создан [26.1] ← [26]


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `RELATED_TO` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=71, offset=71>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 71, 'line': 2, 'column': 71}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (z:Zettel {zettel_id: $zettel_id, user_id: $user_id})-[:RELATED_TO*1..1]-(related:Zettel {user_id: $user_id})\n        WHERE related.zettel_id <> $zettel_id\n        RETURN DISTINCT related as z\n        LIMIT 10\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N51', s

  [Neo4j] Создан [27]: Петров должен согласовать с DevOps переход staging...
  [Neo4j] Создан [27.1] ← [27]
  [Neo4j] Создан [27.1a] ← [27.1]
  [Neo4j] Создан [28]: Существует риск, что конкурент BrightApps выпустит...
  [Neo4j] Создан [28.1] ← [28]


Обработка заметок:  70%|███████   | 7/10 [02:28<01:04, 21.53s/it]

# Retriever inference 

In [45]:
# Для каждого вопроса из qa_dataset_annotated запускает GraphRetriever
# и сохраняет ranked список retrieved zettel_id.


In [ ]:
ANNOTATED_PATH = "synthetic_datasets/graphrag/qa_dataset_annotated.xlsx"
RETRIEVED_PATH = "synthetic_datasets/graphrag/retriever_results.xlsx"
TEST_USER_ID   = "graphrag_test_user_v9"

In [48]:

def run_retriever():
    df_qa = pd.read_excel(ANNOTATED_PATH)

    # оставляем только вопросы с хотя бы одной релевантной карточкой
    df_qa = df_qa[df_qa["n_relevant_zettels"] > 0].copy()
    print(f"Вопросов для оценки retriever: {len(df_qa)}")

    embedding_model = LocalEmbeddingModel()
    client          = get_neo4j_client()
    repository      = ZettelRepository(client)
    retriever       = GraphRetriever(
        embedding_model=embedding_model,
        repository=repository,
    )

    rows = []

    for _, row in tqdm(df_qa.iterrows(), total=len(df_qa), desc="Retriever inference"):
        question = str(row["question"])

        # запуск retriever
        context = retriever.retrieve(
            user_id=TEST_USER_ID,
            query=question,
        )

        # ranked список: entry_points (vector search) → expanded (graph traversal)
        # порядок важен для MRR и NDCG
        retrieved_ranked = [n.zettel_id for n in context.entry_points] + \
                           [n.zettel_id for n in context.expanded_nodes]

        # дедупликация с сохранением порядка
        seen, retrieved_dedup = set(), []
        for zid in retrieved_ranked:
            if zid not in seen:
                seen.add(zid)
                retrieved_dedup.append(zid)

        result_row = row.to_dict()
        result_row["retrieved_zettel_ids_ranked"] = "|".join(retrieved_dedup)
        result_row["n_retrieved"]    = len(retrieved_dedup)
        result_row["n_entry_points"] = len(context.entry_points)
        result_row["n_expanded"]     = len(context.expanded_nodes)
        rows.append(result_row)

    df_results = pd.DataFrame(rows)
    df_results.to_excel(RETRIEVED_PATH, index=False)
    print(f"Результаты retriever → {RETRIEVED_PATH}")
    return df_results


if __name__ == "__main__":
    run_retriever()

FileNotFoundError: [Errno 2] No such file or directory: 'synthetic_datasets/graphrag/qa_dataset_annotated.xlsx'

##  Генерируем синтетические Q&A

In [ ]:
import os
import uuid
import pandas as pd
import json
from typing import Optional
from tqdm import tqdm
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, util

from zettelkasten.atomizer import NoteAtomizer
from zettelkasten.linker import GraphLinker, LocalEmbeddingModel
from storage.neo4j.client import get_neo4j_client
from storage.neo4j.schema import init_schema
from storage.neo4j.repository import ZettelRepository

load_dotenv()

True

In [ ]:
class QAPair(BaseModel):
    question: str = Field(
        description="Натуральный вопрос топ-менеджера к своей базе знаний"
    )
    reference_answer: str = Field(
        description="Полный эталонный ответ на основе заметки"
    )
    question_type: str = Field(
        description="Тип: factual / summary / action_items / risk / decision / detail"
    )
    key_concepts: list[str] = Field(
        description="3-7 ключевых концептов для оценки покрытия ответа"
    )
    relevant_sentences: list[str] = Field(
        description=(
            "Список предложений/фраз из исходной заметки, "
            "которые СОДЕРЖАТ ответ на вопрос. "
            "Это будут 'якоря' для поиска релевантных Zettel-карточек."
        )
    )


QA_SYSTEM_PROMPT = """Ты — топ-менеджер, который пользуется персональной системой управления знаниями.
Тебе показывают одну из твоих старых заметок.

Твоя задача — придумать ОДИН реалистичный вопрос, который ты мог бы задать системе СПУСТЯ НЕКОТОРОЕ ВРЕМЯ
после того, как сделал эту заметку. Вопрос должен:
- Звучать как натуральный вопрос к своей базе знаний
- Быть специфичным для данной заметки (содержать уникальные детали: имена, проекты, цифры)
- Иметь однозначный ответ в тексте заметки
- Не быть слишком простым и не слишком широким
- Вопрос не должен дословно цитировать текст заметки
- Быть верхнеуровневым — как будто ты забыл детали, но помнишь контекст

Также предоставь:
- reference_answer: точный и полный ответ, основанный ТОЛЬКО на тексте заметки
- question_type: тип вопроса (factual / summary / action_items / risk / decision / detail)
- key_concepts: 3-7 ключевых концептов из заметки, которые должны присутствовать в ответе
- relevant_sentences: список конкретных предложений/фраз из исходной заметки,
  которые содержат ответ на вопрос (дословные цитаты из текста заметки).
  Это критически важно для последующей оценки retriever — по этим предложениям
  мы определим, какие Zettel-карточки являются релевантными для данного вопроса.
"""

QA_USER_PROMPT = """Вот текст заметки:

---
{note_text}
---

Сгенерируй вопрос, эталонный ответ и relevant_sentences для оценки системы."""



In [ ]:
class QADatasetGenerator:
    def __init__(self, model_name: str = "openai/gpt-5.1"):
        self.llm = ChatOpenAI(
            model=model_name,
            api_key=os.getenv("LLM_API_KEY"),
            base_url=os.getenv("LLM_BASE_URL"),
            temperature=0.7,
        ).with_structured_output(QAPair)

    def generate_qa(self, note_text: str) -> Optional[QAPair]:
        try:
            return self.llm.invoke([
                SystemMessage(content=QA_SYSTEM_PROMPT),
                HumanMessage(content=QA_USER_PROMPT.format(note_text=note_text)),
            ])
        except Exception as e:
            print(f"Ошибка генерации: {e}")
            return None



In [ ]:


def generate_qa_dataset(
    notes_path: str = "synthetic_datasets/notes_dataset_v1.xlsx",
    # linker_sample_path: str = "synthetic_datasets/atomizer/thoughts_google_gemini-2.5-flash_500_samples.xlsx",
    output_path: str = "synthetic_datasets/graphrag/qa_dataset_raw.xlsx",
    model_name: str = "openai/gpt-4.1",
):
    # загружаем заметки
    df_notes = pd.read_excel(notes_path)
    # # linker_sample = pd.read_excel(linker_sample_path)
    # df_notes = df_notes[df_notes["note_id"].isin(linker_sample["note_id"])].copy()

    print(f"Заметок для генерации Q&A: {len(df_notes)}")

    generator = QADatasetGenerator(model_name=model_name)
    rows = []

    for _, row in tqdm(df_notes.iterrows(), total=len(df_notes), desc="Генерация Q&A"):
        qa = generator.generate_qa(row["note_text"])
        if qa is None:
            continue

        rows.append({
            "qa_id":             str(uuid.uuid4()),
            "note_id":           row["note_id"],
            "note_text":         row["note_text"],
            "domain":            row.get("domain", ""),
            "style":             row.get("style", ""),
            "question":          qa.question,
            "reference_answer":  qa.reference_answer,
            "question_type":     qa.question_type,
            "key_concepts":      "|".join(qa.key_concepts),
            # JSON-список предложений из заметки → якоря для поиска relevant zettel_id
            "relevant_sentences": "|||".join(qa.relevant_sentences),
        })

    df_out = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_out.to_excel(output_path, index=False)
    print(f"Сохранено {len(df_out)} Q&A пар → {output_path}")
    return df_out


if __name__ == "__main__":
    generate_qa_dataset()

Заметок для генерации Q&A: 200


Генерация Q&A: 100%|██████████| 200/200 [12:29<00:00,  3.75s/it]

Сохранено 200 Q&A пар → synthetic_datasets/graphrag/qa_dataset_raw.xlsx


## Прогон заметок через Atomizer + Linker → получение zettel_id

После генерации Q&A нужно прогнать те же самые заметки через полный пайплайн и сохранить маппинг note_id → [zettel_id_1, zettel_id_2, ...]

In [33]:
QA_DATASET_PATH = "synthetic_datasets/graphrag/qa_dataset_raw.xlsx"
MAPPING_OUTPUT  = "synthetic_datasets/graphrag/note_to_zettel_mapping.json"

# фиксированный тестовый user_id — изолированный граф для тестов
TEST_USER_ID = "graphrag_test_user_v2"

In [8]:
def build_graph_and_collect_ids():
    df_qa = pd.read_excel(QA_DATASET_PATH)
    # дедупликация: каждую заметку прогоняем один раз
    df_notes = df_qa.drop_duplicates(subset=["note_id"])[["note_id", "note_text"]]

    print(f"Заметок для обработки: {len(df_notes)}")

    # инициализация компонентов
    embedding_model = LocalEmbeddingModel()
    client = get_neo4j_client()
    init_schema(client)
    repository = ZettelRepository(client)

    atomizer = NoteAtomizer()
    linker = GraphLinker(
        embedding_model=embedding_model,
        repository=repository,
    )
    print(f'atomizer model name: {atomizer.model_name}')
    print(f'linker model name: {linker.model_name}')

    # маппинг: note_id → список zettel_id карточек
    note_to_zettels: dict[str, list[str]] = {}

    for _, row in tqdm(df_notes.iterrows(), total=len(df_notes), desc="Atomizer + Linker"):
        note_id   = str(row["note_id"])
        note_text = str(row["note_text"])

        # atomizer: текст → список ZettelCard
        current_max_root = repository.get_max_root_id(TEST_USER_ID)
        cards = atomizer.atomize(note_text, current_db_max_root_id=current_max_root)

        if isinstance(cards, str):
            # ошибка atomizer
            print(f"[WARN] Atomizer error для note_id={note_id}: {cards}")
            note_to_zettels[note_id] = []
            continue

        # linker: ZettelCard → ZettelNode в Neo4j
        results = linker.link_and_insert(TEST_USER_ID, cards)

        # собираем zettel_id всех вставленных карточек
        zettel_ids = [
            res.card.zettel_id
            for res in results
            if res.card and res.card.zettel_id
        ]

        note_to_zettels[note_id] = zettel_ids
        print(f"  note_id={note_id}: {len(zettel_ids)} карточек → {zettel_ids[:3]}...")

    # сохраняем маппинг
    os.makedirs(os.path.dirname(MAPPING_OUTPUT), exist_ok=True)
    with open(MAPPING_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(note_to_zettels, f, ensure_ascii=False, indent=2)

    print(f"\nМаппинг сохранён → {MAPPING_OUTPUT}")
    print(f"Всего заметок: {len(note_to_zettels)}")
    print(f"Всего карточек: {sum(len(v) for v in note_to_zettels.values())}")

    return note_to_zettels


if __name__ == "__main__":
    build_graph_and_collect_ids()

NameError: name 'QA_DATASET_PATH' is not defined

## Разметка relevant_zettel_ids для каждого вопроса

In [ ]:
# scripts/annotate_relevant_zettels.py
"""
Для каждого Q&A вопроса определяет список relevant_zettel_ids.

Два метода:
  1. LLM-разметка (точнее, но дороже) — GPT сверяет content карточки с вопросом
  2. Sentence-similarity (быстрее) — косинусное сходство content vs relevant_sentences

Используем комбинацию: similarity как фильтр, LLM как верификатор.
"""


QA_DATASET_PATH   = "synthetic_datasets/graphrag/qa_dataset_raw.xlsx"
MAPPING_PATH      = "synthetic_datasets/graphrag/note_to_zettel_mapping.json"
OUTPUT_PATH       = "synthetic_datasets/graphrag/qa_dataset_annotated.xlsx"
TEST_USER_ID      = "graphrag_test_user_v1"

# порог косинусного сходства: content карточки vs relevant_sentence
SIMILARITY_THRESHOLD_ANNOTATION = 0.60  


# ── Pydantic схема LLM-верификатора ───────────────────────────────────────────

class RelevanceVerdict(BaseModel):
    is_relevant: bool = Field(
        description="True если карточка содержит информацию, отвечающую на вопрос"
    )
    reason: str = Field(description="Краткое обоснование (1 предложение)")


VERIFY_SYSTEM_PROMPT = """Ты — эксперт по оценке релевантности.
Тебе дан вопрос пользователя и содержимое одной карточки из базы знаний.
Определи: содержит ли эта карточка информацию, которая помогает ответить на вопрос?

Отвечай строго: is_relevant=True или False."""

VERIFY_USER_PROMPT = """Вопрос: {question}

Содержимое карточки: {content}

Карточка релевантна вопросу?"""


class RelevanceAnnotator:
    def __init__(
        self,
        embedding_model_name: str = "intfloat/multilingual-e5-base",
        llm_model_name: str = "openai/gpt-4.1",
        sim_threshold: float = SIMILARITY_THRESHOLD_ANNOTATION,
    ):
        self.sim_threshold = sim_threshold

        # локальная модель для быстрого pre-filtering
        self.embed_model = SentenceTransformer(embedding_model_name)

        # LLM для верификации пограничных случаев
        self.llm = ChatOpenAI(
            model=llm_model_name,
            api_key=os.getenv("LLM_API_KEY"),
            base_url=os.getenv("LLM_BASE_URL"),
            temperature=0.0,
        ).with_structured_output(RelevanceVerdict)

    def _embed(self, texts: list[str]) -> np.ndarray:
        prefixed = [f"passage: {t}" for t in texts]
        return self.embed_model.encode(prefixed, normalize_embeddings=True, show_progress_bar=False)

    def annotate(
        self,
        question: str,
        relevant_sentences: list[str],
        candidate_cards: list[dict],  # list of {"zettel_id": str, "content": str}
    ) -> list[str]:
        """
        Возвращает список relevant_zettel_ids для данного вопроса.

        Алгоритм:
        1. Эмбеддим relevant_sentences и content всех карточек заметки
        2. Если max cosine sim(content, relevant_sentences) >= threshold → relevant
        3. Пограничная зона [0.45, threshold) → LLM верификация
        4. < 0.45 → не релевантна
        """
        if not candidate_cards or not relevant_sentences:
            return []

        contents = [c["content"] for c in candidate_cards]
        zettel_ids = [c["zettel_id"] for c in candidate_cards]

        # эмбеддинги
        sent_embs  = self._embed(relevant_sentences)     # (n_sentences, dim)
        card_embs  = self._embed(contents)               # (n_cards, dim)

        # для каждой карточки: максимальное сходство с любым relevant_sentence
        sim_matrix = util.cos_sim(card_embs, sent_embs).numpy()  # (n_cards, n_sentences)
        max_sims   = sim_matrix.max(axis=1)                       # (n_cards,)

        relevant_ids  = []
        borderline    = []  # индексы для LLM-проверки

        for idx, (zettel_id, max_sim) in enumerate(zip(zettel_ids, max_sims)):
            if max_sim >= self.sim_threshold:
                relevant_ids.append(zettel_id)
            elif max_sim >= 0.45:
                borderline.append((idx, zettel_id, max_sim))

        # LLM верификация для пограничных случаев
        for idx, zettel_id, sim in borderline:
            try:
                verdict = self.llm.invoke([
                    SystemMessage(content=VERIFY_SYSTEM_PROMPT),
                    HumanMessage(content=VERIFY_USER_PROMPT.format(
                        question=question,
                        content=contents[idx],
                    )),
                ])
                if verdict.is_relevant:
                    relevant_ids.append(zettel_id)
            except Exception as e:
                print(f"LLM verify error: {e}")

        return relevant_ids


def annotate_dataset():
    df_qa = pd.read_excel(QA_DATASET_PATH)

    with open(MAPPING_PATH, encoding="utf-8") as f:
        note_to_zettels: dict[str, list[str]] = json.load(f)

    # загружаем карточки из Neo4j
    client = get_neo4j_client()
    repo   = ZettelRepository(client)

    # кешируем карточки по zettel_id
    print("Загружаем все карточки из Neo4j...")
    all_zettel_ids = [zid for ids in note_to_zettels.values() for zid in ids]
    
    zettel_cache: dict[str, str] = {}  # zettel_id → content
    for zid in tqdm(all_zettel_ids, desc="Загрузка карточек"):
        node = repo.get_by_id(TEST_USER_ID, zid)
        if node:
            zettel_cache[zid] = node.content

    annotator = RelevanceAnnotator()
    results   = []

    for _, row in tqdm(df_qa.iterrows(), total=len(df_qa), desc="Разметка relevant_zettel_ids"):
        note_id   = str(row["note_id"])
        question  = str(row["question"])

        # предложения-якоря из заметки
        rel_sentences = [
            s.strip()
            for s in str(row["relevant_sentences"]).split("|||")
            if s.strip()
        ]

        # все карточки данной заметки
        candidate_zettel_ids = note_to_zettels.get(note_id, [])
        candidate_cards = [
            {"zettel_id": zid, "content": zettel_cache[zid]}
            for zid in candidate_zettel_ids
            if zid in zettel_cache
        ]

        # разметка
        relevant_ids = annotator.annotate(question, rel_sentences, candidate_cards)

        result_row = row.to_dict()
        result_row["all_zettel_ids"]      = "|".join(candidate_zettel_ids)   # все карточки заметки
        result_row["relevant_zettel_ids"] = "|".join(relevant_ids)           # релевантные для вопроса
        result_row["n_total_zettels"]     = len(candidate_zettel_ids)
        result_row["n_relevant_zettels"]  = len(relevant_ids)
        results.append(result_row)

    df_out = pd.DataFrame(results)
    df_out.to_excel(OUTPUT_PATH, index=False)
    print(f"\nАннотированный датасет → {OUTPUT_PATH}")
    print(df_out[["note_id", "question_type", "n_total_zettels", "n_relevant_zettels"]].describe())
    return df_out


if __name__ == "__main__":
    annotate_dataset()

# to_fix

In [2]:
import os
import sys
import uuid
import json
import ast
import time
import logging
from pathlib import Path
from datetime import datetime
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

from zettelkasten.linker import LocalEmbeddingModel

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)
print('✅ imports ok')

/Users/nastya/github/Executive_Exocortex/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ imports ok


In [6]:
GENERATION_MODEL      = 'openai/gpt-5.1'
EVAL_USER_ID          = 'linker_eval_gpt-5.2'
MAX_WORKERS           = 15
GT_SIM_THRESHOLD      = 0.60    # порог similarity для zettel → note mapping

NOTES_PATH    = Path('synthetic_datasets/notes_dataset_v1.xlsx')
LINKER_PATH   = Path('synthetic_datasets/linker/linker_google_gemini-2.5-flash.xlsx')
OUTPUT_PATH   = Path('synthetic_datasets/rag_qa_dataset.xlsx')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'Paths configured. GT threshold: {GT_SIM_THRESHOLD}')

Paths configured. GT threshold: 0.6


In [7]:
df_notes  = pd.read_excel(NOTES_PATH)
df_notes.head(3)

,№,note_id,note_text,num_sentences,actual_sentences,domain,style,created_at
0,1,c09941ef-4576-4710-936d-dc12911f0ab7,Думаю по реструктуризации после разговора с Ма...,4,4,реструктуризация компании,рабочие мысли,2026-06-27 11:49:56
1,2,1d4f8aa3-9dc0-4898-b324-3b3c61a67436,Разобрал цифры по благотворительному проекту С...,3,3,благотворительный проект,анализ ситуации,2026-06-27 11:49:56
2,3,3a046e1a-56fb-4abb-afc4-cb65d85d3a24,Звонок с клиентом Северсталь Диджитал по проек...,6,6,работа с клиентами,записи после звонка,2026-06-27 11:49:58


In [9]:

df_linker = pd.read_excel(LINKER_PATH)
f_linker = df_linker[df_linker['user_id'] == EVAL_USER_ID].reset_index(drop=True)
df_linker.head(3)

,zettel_id,luhmann_id,content,thought_type,tags,is_root_topic,user_id,embedding,created_at,updated_at,similarity,action,reasoning,candidates_found,target_zettel_id,latency_sec,model_name
0,58321c67-bb3d-4804-aacd-534eb430b537,1,Состоялся звонок с Мариной из SkyLearn и Томом...,context,"['марина', 'skylearn', 'том_harris', 'проект_с...",True,linker_eval_google_gemini-2.5-flash,"[-0.006099278572946787, 0.08536402881145477, -...",NaN,NaN,NaN,new_root,Нет похожих карточек в графе,0,NaN,245.322719,google/gemini-2.5-flash
1,091f8612-2695-4598-aaac-a570ea2cda13,1.1,Марина из SkyLearn и Том Harris подтвердили же...,fact,"['марина', 'том_harris', 'проект_сова', '120_м...",False,linker_eval_google_gemini-2.5-flash,"[0.00489732064306736, 0.07277049869298935, -0....",NaN,NaN,NaN,child_of,Дочерняя карточка внутри текущего сообщения.,0,NaN,245.322719,google/gemini-2.5-flash
2,b128373b-f539-4a0e-8322-cbeb3d365820,1.1a,Марина из SkyLearn и Том Harris просят предост...,action,"['марина', 'том_harris', 'moodle', 'power_bi',...",False,linker_eval_google_gemini-2.5-flash,"[0.008820367977023125, 0.06661751121282578, -0...",NaN,NaN,NaN,child_of,Дочерняя карточка внутри текущего сообщения.,0,NaN,245.322719,google/gemini-2.5-flash


In [10]:
embedder = LocalEmbeddingModel()

print('Embedding notes...')
note_texts = df_notes['note_text'].tolist()
note_embs  = np.array([embedder.embed_passage(t) for t in tqdm(note_texts, desc='notes')])
print(f'Note embeddings shape: {note_embs.shape}')

2026-06-29 20:05:27,858 | INFO | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-06-29 20:05:28,047 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-29 20:05:28,107 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/modules.json "HTTP/1.1 200 OK"
2026-06-29 20:05:28,299 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-06-29 20:05:28,301 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-29 20:05:28,305 | INFO | Loading SentenceTransformer model from intfloat/multilingual-e5-base.
2026-06-29 20:05:28,486 | INFO | HTTP Request: HEAD https://hugging

[EmbeddingModel] intfloat/multilingual-e5-base загружена за 7.3с (dim=768, device=mps)
Embedding notes...


notes: 100%|██████████| 200/200 [00:09<00:00, 21.10it/s]

Note embeddings shape: (200, 768)


In [5]:
def parse_embedding(val):
    """Парсим embedding из строки / списка / None."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    if isinstance(val, list):
        return np.array(val, dtype=np.float32)
    if isinstance(val, str):
        try:
            return np.array(ast.literal_eval(val), dtype=np.float32)
        except Exception:
            return None
    return None

raw_embs = [parse_embedding(e) for e in df_linker['embedding']]
valid_mask = [e is not None for e in raw_embs]
df_linker = df_linker[valid_mask].reset_index(drop=True)
zettel_embs = np.array([e for e in raw_embs if e is not None])

# Нормализуем zettel embeddings (на случай если они не нормализованы)
norms = np.linalg.norm(zettel_embs, axis=1, keepdims=True)
zettel_embs = zettel_embs / (norms + 1e-10)

print(f'Zettel embeddings shape: {zettel_embs.shape}')
print(f'Filtered zettels with valid embeddings: {len(df_linker)}')

Zettel embeddings shape: (484, 768)
Filtered zettels with valid embeddings: 484


In [6]:
# Матрица сходства: (n_zettels x n_notes)
sim_matrix = zettel_embs @ note_embs.T
print(f'Similarity matrix shape: {sim_matrix.shape}')

# Для каждого zettel → best matching note
best_note_idx = sim_matrix.argmax(axis=1)
best_note_sim = sim_matrix.max(axis=1)

df_linker['matched_note_id']  = None
df_linker['matched_note_sim'] = best_note_sim

for i in range(len(df_linker)):
    if best_note_sim[i] >= GT_SIM_THRESHOLD:
        df_linker.at[i, 'matched_note_id'] = df_notes.iloc[best_note_idx[i]]['note_id']

matched = df_linker['matched_note_id'].notna().sum()
print(f'Zettels matched to notes (sim >= {GT_SIM_THRESHOLD}): {matched} / {len(df_linker)}')

# note_id → список zettel_ids (ground truth для retrieval)
note_to_gt_zettels = defaultdict(list)
for _, row in df_linker.iterrows():
    if row['matched_note_id'] is not None:
        note_to_gt_zettels[row['matched_note_id']].append(row['zettel_id'])

unique_notes_with_gt = len(note_to_gt_zettels)
print(f'Notes with at least 1 matched zettel: {unique_notes_with_gt}')
print(f'Avg GT zettels per note: {np.mean([len(v) for v in note_to_gt_zettels.values()]):.1f}')

Similarity matrix shape: (484, 200)
Zettels matched to notes (sim >= 0.6): 484 / 484
Notes with at least 1 matched zettel: 131
Avg GT zettels per note: 3.7


In [7]:
QA_SYSTEM_PROMPT = """Ты — топ-менеджер, который пользуется персональной системой управления знаниями.
Тебе показывают одну из твоих старых заметок.

Твоя задача — придумать ОДИН реалистичный вопрос, который ты мог бы задать системе СПУСТЯ НЕКОТОРОЕ ВРЕМЯ
после того, как сделал эту заметку. Вопрос должен:
- Звучать как натуральный вопрос к своей базе знаний («напомни», «что мы решали», «кто отвечал», «какие цифры» и т.д.)
- Быть специфичным для данной заметки (содержать уникальные детали: имена, проекты, цифры)
- Иметь однозначный ответ в тексте заметки
- Не быть слишком простым («о чём эта заметка?») и не слишком широким

Также предоставь:
- reference_answer: точный и полный ответ на вопрос, основанный ТОЛЬКО на тексте заметки
- question_type: тип вопроса (factual / summary / action_items / risk / decision / detail)
- key_concepts: список 3-7 ключевых концептов из заметки, которые должны присутствовать в ответе"""


class QAPair(BaseModel):
    question: str = Field(description="Натуральный вопрос топ-менеджера к своей базе знаний")
    reference_answer: str = Field(description="Полный эталонный ответ на основе заметки")
    question_type: str = Field(description="Тип: factual / summary / action_items / risk / decision / detail")
    key_concepts: list[str] = Field(description="3-7 ключевых концептов для оценки покрытия ответа")


print('QA prompt schema ready')

QA prompt schema ready


In [8]:
save_lock = Lock()
results   = []


def generate_single_qa(row: dict) -> dict | None:
    note_id   = row['note_id']
    note_text = row['note_text']
    domain    = row.get('domain', '')
    style     = row.get('style', '')

    llm = ChatOpenAI(
        model=GENERATION_MODEL,
        temperature=0.7,
        api_key=os.getenv('LLM_API_KEY'),
        base_url=os.getenv('LLM_BASE_URL'),
    )

    try:
        qa: QAPair = llm.with_structured_output(QAPair).invoke([
            SystemMessage(content=QA_SYSTEM_PROMPT),
            HumanMessage(content=f"Заметка:\n{note_text}"),
        ])
        return {
            'question_id':             f'Q-{uuid.uuid4().hex[:8].upper()}',
            'source_note_id':          note_id,
            'note_text':               note_text,
            'note_domain':             domain,
            'note_style':              style,
            'question':                qa.question,
            'reference_answer':        qa.reference_answer,
            'question_type':           qa.question_type,
            'key_concepts':            json.dumps(qa.key_concepts, ensure_ascii=False),
            'ground_truth_zettel_ids': json.dumps(
                note_to_gt_zettels.get(note_id, []), ensure_ascii=False
            ),
            'n_gt_zettels':            len(note_to_gt_zettels.get(note_id, [])),
            'generated_at':            datetime.now().isoformat(),
            'model_name':              GENERATION_MODEL,
        }
    except Exception as e:
        logger.error(f'Error generating QA for note {note_id}: {e}')
        return None


rows = df_notes.to_dict('records')

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(generate_single_qa, r): r['note_id'] for r in rows}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Generating Q&A'):
        res = future.result()
        if res is not None:
            with save_lock:
                results.append(res)

df_qa = pd.DataFrame(results)
df_qa['№'] = range(1, len(df_qa) + 1)
print(f'Generated: {len(df_qa)} Q&A pairs')
df_qa.head(3)

Generating Q&A:   0%|          | 0/200 [00:00<?, ?it/s]2026-06-28 17:13:16,608 | INFO | HTTP Request: POST https://litellm.tokengate.ru/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-06-28 17:13:16,610 | INFO | Retrying request to /chat/completions in 0.421826 seconds
2026-06-28 17:13:16,619 | INFO | HTTP Request: POST https://litellm.tokengate.ru/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-06-28 17:13:16,620 | INFO | Retrying request to /chat/completions in 0.464909 seconds
2026-06-28 17:13:17,205 | INFO | HTTP Request: POST https://litellm.tokengate.ru/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-06-28 17:13:17,205 | INFO | Retrying request to /chat/completions in 0.840266 seconds
2026-06-28 17:13:17,372 | INFO | HTTP Request: POST https://litellm.tokengate.ru/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-06-28 17:13:17,373 | INFO | Retrying request to /chat/completions in 0.909840 seconds
2026-06-28 17:13:18,281 | INFO | HTTP R

Generated: 199 Q&A pairs


,question_id,source_note_id,note_text,note_domain,note_style,question,reference_answer,question_type,key_concepts,ground_truth_zettel_ids,n_gt_zettels,generated_at,model_name,№
0,Q-0A1D247E,33ba4155-d599-4f78-b9f2-f18f10c6bd4a,Звонок с Андреем и Michael из DHL по поставкам...,операционная деятельность,записи после звонка,"Напомни, какой именно лимит на экспресс-достав...",Решили временно поднять лимит на экспресс до 1...,factual,"[""лимит на экспресс"", ""1,8 млн рублей"", ""проек...","[""fa44b3b5-f93c-40ee-bdee-870f0f1a08a1""]",1,2026-06-28T17:13:18.289649,openai/gpt-5.1,1
1,Q-FB4EB54F,142f182e-072c-4364-9058-c71303b5a194,Только что посмотрел с Олей и Mark дашборд в T...,аналитика и метрики,голосовая заметка,"Напомни, какую оценку по потере MRR называла К...",Катя оценивает потенциальное влияние на MRR ка...,factual,"[""Катя"", ""падение конверсии trial→оплата у Сбе...","[""cd4790f7-8b58-4c57-bfc3-13b1d2d3a464"", ""ec29...",5,2026-06-28T17:13:18.299467,openai/gpt-5.1,2
2,Q-8F323939,9c5f97c0-f0f3-4cc0-b272-5ddf4bab53b4,"Думаю по людям в команде Phoenix, нагрузка уже...",управление персоналом,рабочие мысли,"Напомни, к какому сроку мы рисковали сдвинуть ...",Если до 15 июля не закрыть двух backend-разраб...,factual,"[""проект Vega для MTS"", ""срок до 15 июля"", ""дв...",[],0,2026-06-28T17:13:18.446104,openai/gpt-5.1,3


In [11]:
df_qa.to_excel(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')

print('\n=== Dataset Statistics ===')
print(f'Total Q&A pairs:              {len(df_qa)}')
print(f'With GT zettels:              {(df_qa["n_gt_zettels"] > 0).sum()}')
print(f'Avg GT zettels per question:  {df_qa["n_gt_zettels"].mean():.2f}')
print(f'Questions with 0 GT zettels:  {(df_qa["n_gt_zettels"] == 0).sum()}')

print('\nQuestion type distribution:')
print(df_qa['question_type'].value_counts())

# Предупреждение: вопросы без GT zettel-ов будут пропущены в retrieval eval
zero_gt = (df_qa['n_gt_zettels'] == 0).sum()
if zero_gt > 0:
    print(f'\n⚠️  {zero_gt} questions have 0 GT zettels (threshold={GT_SIM_THRESHOLD}).')
    print('Consider lowering GT_SIM_THRESHOLD if too many are missing.')

Saved to synthetic_datasets/rag_qa_dataset.xlsx

=== Dataset Statistics ===
Total Q&A pairs:              199
With GT zettels:              131
Avg GT zettels per question:  2.43
Questions with 0 GT zettels:  68

Question type distribution:
question_type
factual         154
decision         32
action_items      6
risk              4
detail            2
summary           1
Name: count, dtype: int64

⚠️  68 questions have 0 GT zettels (threshold=0.6).
Consider lowering GT_SIM_THRESHOLD if too many are missing.


In [12]:
# Просмотр нескольких примеров
for i, row in df_qa.head(2).iterrows():
    print(f"{'='*80}")
    print(f"[{row['question_type'].upper()}] {row['question']}")
    print(f"\nAnswer: {row['reference_answer'][:300]}...")
    print(f"GT zettels ({row['n_gt_zettels']}): {row['ground_truth_zettel_ids'][:100]}...")

[FACTUAL] Напомни, какой именно лимит на экспресс-доставку мы тогда решили временно поднять в рамках проекта «Север» для клиента ВекторФарм из‑за задержки партии в Риге?

Answer: Решили временно поднять лимит на экспресс до 1,8 млн рублей....
GT zettels (1): ["fa44b3b5-f93c-40ee-bdee-870f0f1a08a1"]...
[FACTUAL] Напомни, какую оценку по потере MRR называла Катя из‑за падения конверсии trial→оплата у СберМаркета в проекте Orion, если тренд не развернуть?

Answer: Катя оценивает потенциальное влияние на MRR как минус 4,5 млн рублей, если тренд падения конверсии не развернуть....
GT zettels (5): ["cd4790f7-8b58-4c57-bfc3-13b1d2d3a464", "ec29692a-3b9b-46af-bc64-7b49e192bdfa", "3a47e2ef-cffb-4835...
